# SVM Sentiment Classifier — Stock Social Media Posts

Predicts sentiment class (`-1` = Bearish, `0` = Neutral, `1` = Bullish) from post text using TF-IDF + LinearSVC.

**Metrics reported per stock**: Accuracy, Confusion Matrix, Precision, Recall, F1 Score, MCC


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    matthews_corrcoef,
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
# ── Configuration ─────────────────────────────────────────────
DATA_DIR = os.path.join("..", "pre-processing")   # path to _final.csv files
STOCKS = ["AAPL", "AMZN", "FB", "NVDA", "TSLA"]
SAMPLE_SIZE = 50000         # rows per stock (stratified)
RANDOM_STATE = 42
TEST_SIZE = 0.20
MAX_TFIDF_FEATURES = 10_000
LABEL_MAP = {-1: "Bearish", 0: "Neutral", 1: "Bullish"}
LABEL_ORDER = [-1, 0, 1]
LABEL_NAMES = ["Bearish", "Neutral", "Bullish"]

## 1 — Load Data & Stratified Sampling

Each stock dataset is sampled to 50K rows (stratified by class) to keep SVM training fast while preserving class proportions.


In [ ]:
# Load each _final.csv → clean → balanced sample (equal per class)
data = {}

for stock in STOCKS:
    path = os.path.join(DATA_DIR, f"{stock}_final.csv")
    print(f"Loading {stock} from {path} ...")
    df = pd.read_csv(path, low_memory=False)
    print(f"  Raw rows: {len(df):,}")
 
    # Balanced sample: equal number per class, capped by the smallest class
    class_counts = df["entities"].value_counts()
    min_class_size = class_counts.min()
    n_per_class = min(min_class_size, SAMPLE_SIZE // 3)  # at most SAMPLE_SIZE/3 per class

    sampled_parts = []
    for label, group in df.groupby("entities"):
        sampled_parts.append(group.sample(n=n_per_class, random_state=RANDOM_STATE))
    df = pd.concat(sampled_parts).reset_index(drop=True)

    print(f"  Sampled rows: {len(df):,}  ({n_per_class:,} per class)")
    print(f"  Class distribution:\n{df['entities'].value_counts().sort_index().to_string()}")
    print()

    data[stock] = df

print("All datasets loaded.")

## 2 — TF-IDF Vectorization + Train/Test Split

We convert `body_no_stopwords` into TF-IDF features (unigrams + bigrams, top 10K features) and split 80/20 for each stock.


In [ ]:
# Vectorize and split each stock
splits = {}  # stock → (X_train, X_test, y_train, y_test, vectorizer)

for stock in STOCKS:
    df = data[stock]
    print(f"Vectorizing {stock} ...")

    vectorizer = TfidfVectorizer(
        max_features=MAX_TFIDF_FEATURES,
        ngram_range=(1, 2),
        sublinear_tf=True,        # dampens term-frequency with 1 + log(tf)
    )

    X = vectorizer.fit_transform(df["body_no_stopwords"])
    y = df["entities"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    splits[stock] = (X_train, X_test, y_train, y_test, vectorizer)
    print(f"  Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}  |  Features: {X_train.shape[1]:,}")

print("\nVectorization complete.")

## 3 — Train LinearSVC & Evaluate Each Stock

For each stock we train a `LinearSVC` with `class_weight='balanced'` and report:

- **Accuracy**
- **Confusion Matrix** (heatmap)
- **Precision, Recall, F1** (per-class + weighted avg)
- **MCC** (Matthews Correlation Coefficient)


In [ ]:
# Train & evaluate each stock individually
results = {}  # stock → dict of metrics

for stock in STOCKS:
    X_train, X_test, y_train, y_test, vec = splits[stock]

    # ── Train ────────────────────────────────────────────────
    clf = LinearSVC(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        dual="auto",
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # ── Metrics ──────────────────────────────────────────────
    acc = accuracy_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    cm = confusion_matrix(y_test, y_pred, labels=LABEL_ORDER)

    results[stock] = {
        "accuracy": acc,
        "f1_weighted": f1_w,
        "f1_macro": f1_m,
        "precision_weighted": prec_w,
        "recall_weighted": rec_w,
        "precision_macro": prec_m,
        "recall_macro": rec_m,
        "mcc": mcc,
        "cm": cm,
        "y_test": y_test,
        "y_pred": y_pred,
    }

    # ── Print report ─────────────────────────────────────────
    print("=" * 60)
    print(f"  {stock} — SVM Results")
    print("=" * 60)
    print(f"  Accuracy : {acc:.4f}")
    print(f"  MCC      : {mcc:.4f}")
    print(f"  F1 (wtd) : {f1_w:.4f}")
    print()
    print(classification_report(
        y_test, y_pred, labels=LABEL_ORDER,
        target_names=LABEL_NAMES, zero_division=0
    ))
    print()

print("All models trained.")

## 4 — Confusion Matrices (all 5 stocks)


In [ ]:
# Plot confusion matrices side-by-side
fig, axes = plt.subplots(1, 5, figsize=(24, 4.5))
fig.suptitle("Confusion Matrices — LinearSVC per Stock", fontsize=14, y=1.02)

for ax, stock in zip(axes, STOCKS):
    cm = results[stock]["cm"]
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
        ax=ax, cbar=False,
    )
    acc = results[stock]["accuracy"]
    ax.set_title(f"{stock}\nAcc={acc:.3f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → confusion_matrices.png")

## 5 — Summary Table (all metrics, all stocks)


In [ ]:
# Build a summary DataFrame for easy comparison
summary_rows = []
for stock in STOCKS:
    r = results[stock]
    summary_rows.append({
        "Stock": stock,
        "Accuracy": round(r["accuracy"], 4),
        "F1 (weighted)": round(r["f1_weighted"], 4),
        "F1 (macro)": round(r["f1_macro"], 4),
        "Precision (weighted)": round(r["precision_weighted"], 4),
        "Recall (weighted)": round(r["recall_weighted"], 4),
        "Precision (macro)": round(r["precision_macro"], 4),
        "Recall (macro)": round(r["recall_macro"], 4),
        "MCC": round(r["mcc"], 4),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Stock")
print("=" * 70)
print("  SUMMARY — LinearSVC Sentiment Classification")
print("=" * 70)
summary_df

In [ ]:
# Bar chart comparing key metrics across stocks
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("SVM Performance Comparison Across Stocks", fontsize=14, y=1.02)

metrics_to_plot = [
    ("Accuracy", "steelblue"),
    ("F1 (weighted)", "darkorange"),
    ("MCC", "seagreen"),
]

for ax, (metric, color) in zip(axes, metrics_to_plot):
    vals = summary_df[metric]
    vals.plot(kind="bar", ax=ax, color=color, edgecolor="black", alpha=0.85)
    ax.set_title(metric, fontsize=12)
    ax.set_ylim(0, max(1.0, vals.max() * 1.15))
    ax.set_ylabel("Score")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("svm_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → svm_comparison.png")

---

# Improvement Experiments

## Experiment 1 — Binary Classification (Bullish vs Bearish)

Drop all Neutral (0) rows and classify only Bullish (1) vs Bearish (-1). This removes the noisy majority class where many posts simply had no sentiment detected.


In [ ]:
# ── Experiment 1: Binary Classification (Bullish vs Bearish) ──────────
BINARY_LABELS = {-1: "Bearish", 1: "Bullish"}
BINARY_ORDER = [-1, 1]
BINARY_NAMES = ["Bearish", "Bullish"]

binary_results = {}

for stock in STOCKS:
    # Filter to only Bearish and Bullish
    df_bin = data[stock][data[stock]["entities"].isin([-1, 1])].copy()
    print(f"{stock}: {len(df_bin):,} rows (Bearish={sum(df_bin['entities']==-1):,}, Bullish={sum(df_bin['entities']==1):,})")

    # TF-IDF
    vec_bin = TfidfVectorizer(max_features=MAX_TFIDF_FEATURES, ngram_range=(1, 2), sublinear_tf=True)
    X_bin = vec_bin.fit_transform(df_bin["body_no_stopwords"])
    y_bin = df_bin["entities"].values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_bin, y_bin, test_size=TEST_SIZE, stratify=y_bin, random_state=RANDOM_STATE
    )

    # Train
    clf_bin = LinearSVC(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE, dual="auto")
    clf_bin.fit(X_tr, y_tr)
    y_pr = clf_bin.predict(X_te)

    # Metrics
    acc = accuracy_score(y_te, y_pr)
    mcc = matthews_corrcoef(y_te, y_pr)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_te, y_pr, average="weighted", zero_division=0)
    cm = confusion_matrix(y_te, y_pr, labels=BINARY_ORDER)

    binary_results[stock] = {
        "accuracy": acc, "f1_weighted": f1_w, "mcc": mcc, "cm": cm,
        "precision_weighted": prec_w, "recall_weighted": rec_w,
        "y_test": y_te, "y_pred": y_pr,
    }

    print(f"  Accuracy: {acc:.4f}  |  F1(wtd): {f1_w:.4f}  |  MCC: {mcc:.4f}")
    print(classification_report(y_te, y_pr, labels=BINARY_ORDER, target_names=BINARY_NAMES, zero_division=0))
    print()

# ── Confusion matrices ──
fig, axes = plt.subplots(1, 5, figsize=(22, 3.5))
fig.suptitle("Exp 1 — Binary (Bullish vs Bearish) Confusion Matrices", fontsize=13, y=1.04)
for ax, stock in zip(axes, STOCKS):
    cm = binary_results[stock]["cm"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", xticklabels=BINARY_NAMES,
                yticklabels=BINARY_NAMES, ax=ax, cbar=False)
    ax.set_title(f"{stock}\nAcc={binary_results[stock]['accuracy']:.3f}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("exp1_binary_cm.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Comparison table ──
comp_rows = []
for stock in STOCKS:
    comp_rows.append({
        "Stock": stock,
        "3-Class Acc": results[stock]["accuracy"],
        "Binary Acc": binary_results[stock]["accuracy"],
        "3-Class F1": results[stock]["f1_weighted"],
        "Binary F1": binary_results[stock]["f1_weighted"],
        "3-Class MCC": results[stock]["mcc"],
        "Binary MCC": binary_results[stock]["mcc"],
    })
comp_df = pd.DataFrame(comp_rows).set_index("Stock")
print("=" * 70)
print("  3-Class vs Binary Comparison")
print("=" * 70)
comp_df